# DEAI-opdracht – Lineaire Regressie met Ames Housing

In deze notebook voorspel ik **SalePrice** met een **lineair regressiemodel** op basis van de dataset uit **AmesHousing.xlsx**.

## Mijn gekozen target en top 3 features
- **Target:** `SalePrice`
- **Top 3 verwachte voorspellers:**
  1. `Overall Qual`
  2. `Gr Liv Area`
  3. `Neighborhood` *(categorisch)*

Waarom deze 3?
- `Overall Qual` zegt iets over de algemene kwaliteit van het huis.
- `Gr Liv Area` zegt iets over de woonoppervlakte.
- `Neighborhood` zegt iets over de ligging/wijk, en dat heeft vaak veel invloed op de prijs.


In [1]:
# Dit blok laadt de libraries die ik nodig heb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)


In [2]:
# Dit blok leest beide tabjes uit het Excel-bestand in als DataFrames
bestand = "AmesHousing.xlsx"

df = pd.read_excel(bestand, sheet_name="AmesHousing")
data_dictionary = pd.read_excel(bestand, sheet_name="Data Dictionary")

print("Vorm van de dataset:", df.shape)
display(df.head())
display(data_dictionary)


Vorm van de dataset: (2930, 12)


,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,NAmes,1Story
1,2,105000,yes,5,896,882.0,11622,1961,1,2,NAmes,1Story
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,NAmes,1Story
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,NAmes,1Story
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,Gilbert,2Story


,Variabele,Betekenis
0,ID,"Uniek nummer per huis, te vergelijken met een ..."
1,SalePrice,Verkoopprijs van het huis (in dollars: USD)
2,Garage,Geeft weer of het huis wel/geen garage bevat
3,Overall Qual,Algemene kwaliteit van materialen en afwerking...
4,Gr Liv Area,Woonoppervlak boven de grond (square feet)
5,Total Bsmt SF,Totale oppervlakte van de kelder
6,Lot Area,Grootte van het perceel (square feet)
7,Year Built,Bouwjaar van het huis
8,Full Bath,Aantal volledige badkamers
9,Bedroom AbvGr,Aantal slaapkamers boven de grond


In [3]:
# Dit blok controleert of elke Excel-kolom goed is ingelezen
print("Kolommen in de DataFrame:")
print(df.columns.tolist())


Kolommen in de DataFrame:
['ID', 'SalePrice', 'Garage', 'Overall Qual', 'Gr Liv Area', 'Total Bsmt SF', 'Lot Area', 'Year Built', 'Full Bath', 'Bedroom AbvGr', 'Neighborhood', 'House Style']


## Stap 3 – Target en eerste featurekeuze

Uit de Data Dictionary blijkt dat:
- **`SalePrice`** de verkoopprijs is, dus dat is de **targetvariabele**.
- Ik start met deze 3 features:
  - `Overall Qual`
  - `Gr Liv Area`
  - `Neighborhood` *(categorisch, dus deze ga ik one-hot encoden)*

> Extra opmerking: `SGDRegressor` is een lineair regressiemodel dat leert in kleine stapjes. Daardoor kan ik ook experimenteren met dingen zoals **epochs** (`max_iter`) en **learning rate** (`eta0` / `learning_rate`).


In [4]:
# Dit blok laat zien welke instellingen (hyperparameters) het model heeft
# In Jupyter kun je hiermee de help-functie van Python gebruiken
help(SGDRegressor)


Help on class SGDRegressor in module sklearn.linear_model._stochastic_gradient:

class SGDRegressor(BaseSGDRegressor)
 |  SGDRegressor(
 |      loss='squared_error',
 |      *,
 |      penalty='l2',
 |      alpha=0.0001,
 |      l1_ratio=0.15,
 |      fit_intercept=True,
 |      max_iter=1000,
 |      tol=0.001,
 |      shuffle=True,
 |      verbose=0,
 |      epsilon=0.1,
 |      random_state=None,
 |      learning_rate='invscaling',
 |      eta0=0.01,
 |      power_t=0.25,
 |      early_stopping=False,
 |      validation_fraction=0.1,
 |      n_iter_no_change=5,
 |      warm_start=False,
 |      average=False
 |  )
 |
 |  Linear model fitted by minimizing a regularized empirical loss with SGD.
 |
 |  SGD stands for Stochastic Gradient Descent: the gradient of the loss is
 |  estimated each sample at a time and the model is updated along the way with
 |  a decreasing strength schedule (aka learning rate).
 |
 |  The regularizer is a penalty added to the loss function that shrinks mode

In [5]:
# Dit blok kiest de eerste features en splitst de data verticaal in X en y
gekozen_features = ["Overall Qual", "Gr Liv Area", "Neighborhood"]
target = "SalePrice"

X = df[gekozen_features]   # Dit zijn de invoerfeatures
y = df[target]             # Dit is de target die ik wil voorspellen

print("Vorm van X:", X.shape)
print("Vorm van y:", y.shape)


Vorm van X: (2930, 3)
Vorm van y: (2930,)


In [6]:
# Dit blok splitst de data horizontaal in train en test
# Zo krijg ik 4 stukken: X_train, X_test, y_train en y_test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


X_train: (2344, 3)
X_test : (586, 3)
y_train: (2344,)
y_test : (586,)


In [7]:
# Dit blok bepaalt welke kolommen numeriek en categorisch zijn
numerieke_kolommen = X_train.select_dtypes(include=["number"]).columns.tolist()
categorische_kolommen = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerieke kolommen:", numerieke_kolommen)
print("Categorische kolommen:", categorische_kolommen)


Numerieke kolommen: ['Overall Qual', 'Gr Liv Area']
Categorische kolommen: ['Neighborhood']


In [8]:
# Dit blok maakt de voorbereiding van de data klaar
# - numerieke kolommen: missende waarden opvullen + schalen
# - categorische kolommen: missende waarden opvullen + one-hot encoden
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerieke_kolommen
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorische_kolommen
        )
    ]
)


In [9]:
# Dit blok maakt het eerste lineaire regressiemodel aan
# Gekozen hyperparameters:
# - max_iter = aantal leer-rondes / epochs
# - eta0 = begin learning rate
# - learning_rate = manier waarop de learning rate wordt aangepast
eerste_model = SGDRegressor(
    max_iter=1000,
    eta0=0.01,
    learning_rate="invscaling",
    alpha=0.0001,
    penalty="l2",
    random_state=42
)

# Dit blok zet preprocessing en model in één pipeline
eerste_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", eerste_model)
])


In [10]:
# Dit blok traint het model op de trainingsdata
eerste_pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [11]:
# Dit blok maakt voorspellingen op de testdata
eerste_voorspellingen = eerste_pipeline.predict(X_test)

# Dit blok rekent de evaluatiemetrieken uit
eerste_mae = mean_absolute_error(y_test, eerste_voorspellingen)
eerste_mse = mean_squared_error(y_test, eerste_voorspellingen)
eerste_rmse = np.sqrt(eerste_mse)
eerste_r2 = r2_score(y_test, eerste_voorspellingen)

eerste_resultaten = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Waarde": [eerste_mae, eerste_mse, eerste_rmse, eerste_r2]
})

display(eerste_resultaten)


,Metric,Waarde
0,MAE,2.560389e+04
1,MSE,1.582183e+09
2,RMSE,3.977666e+04
3,R2,8.026600e-01


## Stap 7 – Experimenteren

Hieronder maak ik meerdere experimenten.  
Ik verander:
- de **features**
- het aantal **epochs** (`max_iter`)
- de **learning rate** (`eta0` en `learning_rate`)

Ik laat alle resultaten staan, zodat je goed kunt uitleggen hoe je tot je beste model bent gekomen.


In [12]:
# Dit blok maakt een handige functie om snel nieuwe experimenten te draaien
def run_experiment(naam, features, max_iter, eta0, learning_rate, alpha=0.0001, penalty="l2"):
    # Dit blok kiest X en y
    X = df[features]
    y = df["SalePrice"]

    # Dit blok splitst de data in train en test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Dit blok zoekt numerieke en categorische kolommen
    numerieke_kolommen = X_train.select_dtypes(include=["number"]).columns.tolist()
    categorische_kolommen = X_train.select_dtypes(exclude=["number"]).columns.tolist()

    # Dit blok bereidt de data voor
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numerieke_kolommen
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorische_kolommen
            )
        ]
    )

    # Dit blok maakt het lineaire regressiemodel
    model = SGDRegressor(
        max_iter=max_iter,
        eta0=eta0,
        learning_rate=learning_rate,
        alpha=alpha,
        penalty=penalty,
        random_state=42
    )

    # Dit blok bouwt de pipeline
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Dit blok traint het model
    pipeline.fit(X_train, y_train)

    # Dit blok maakt voorspellingen
    voorspellingen = pipeline.predict(X_test)

    # Dit blok rekent de evaluatiemetrieken uit
    mae = mean_absolute_error(y_test, voorspellingen)
    mse = mean_squared_error(y_test, voorspellingen)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, voorspellingen)

    # Dit blok geeft alle belangrijke resultaten terug
    return {
        "Experiment": naam,
        "Features": ", ".join(features),
        "Aantal features": len(features),
        "max_iter": max_iter,
        "eta0": eta0,
        "learning_rate": learning_rate,
        "alpha": alpha,
        "penalty": penalty,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }


In [13]:
# Dit blok draait meerdere experimenten achter elkaar
experimenten = []

# Initiële run
experimenten.append(
    run_experiment(
        naam="Initiële run",
        features=["Overall Qual", "Gr Liv Area", "Neighborhood"],
        max_iter=1000,
        eta0=0.01,
        learning_rate="invscaling"
    )
)

# Experiment 1: zelfde features, maar andere epochs en learning rate
experimenten.append(
    run_experiment(
        naam="Exp 1: zelfde features, andere hyperparameters",
        features=["Overall Qual", "Gr Liv Area", "Neighborhood"],
        max_iter=2000,
        eta0=0.005,
        learning_rate="adaptive"
    )
)

# Experiment 2: extra numerieke features erbij
experimenten.append(
    run_experiment(
        naam="Exp 2: meer features",
        features=["Overall Qual", "Gr Liv Area", "Neighborhood", "Total Bsmt SF", "Year Built"],
        max_iter=2000,
        eta0=0.005,
        learning_rate="adaptive"
    )
)

# Experiment 3: nog meer features, inclusief extra categorische info
experimenten.append(
    run_experiment(
        naam="Exp 3: nog meer features",
        features=["Overall Qual", "Gr Liv Area", "Neighborhood", "Total Bsmt SF", "Year Built", "Garage", "House Style"],
        max_iter=2000,
        eta0=0.005,
        learning_rate="adaptive"
    )
)

# Experiment 4: bijna alle bruikbare features
experimenten.append(
    run_experiment(
        naam="Exp 4: bijna alle features",
        features=["Garage", "Overall Qual", "Gr Liv Area", "Total Bsmt SF", "Lot Area", "Year Built", "Full Bath", "Bedroom AbvGr", "Neighborhood", "House Style"],
        max_iter=2000,
        eta0=0.005,
        learning_rate="adaptive"
    )
)

# Experiment 5: zelfde features als exp 4, maar andere epochs en learning rate
experimenten.append(
    run_experiment(
        naam="Exp 5: bijna alle features + andere epochs/lr",
        features=["Garage", "Overall Qual", "Gr Liv Area", "Total Bsmt SF", "Lot Area", "Year Built", "Full Bath", "Bedroom AbvGr", "Neighborhood", "House Style"],
        max_iter=3000,
        eta0=0.003,
        learning_rate="adaptive"
    )
)

resultaten_df = pd.DataFrame(experimenten).sort_values("RMSE").reset_index(drop=True)
display(resultaten_df)


/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


,Experiment,Features,Aantal features,max_iter,eta0,learning_rate,alpha,penalty,MAE,MSE,RMSE,R2
0,Exp 4: bijna alle features,"Garage, Overall Qual, Gr Liv Area, Total Bsmt ...",10,2000,0.005,adaptive,0.0001,l2,21603.154972,1.285773e+09,35857.674231,0.839630
1,Exp 5: bijna alle features + andere epochs/lr,"Garage, Overall Qual, Gr Liv Area, Total Bsmt ...",10,3000,0.003,adaptive,0.0001,l2,21618.714862,1.287074e+09,35875.811641,0.839468
2,Exp 3: nog meer features,"Overall Qual, Gr Liv Area, Neighborhood, Total...",7,2000,0.005,adaptive,0.0001,l2,22113.765090,1.317108e+09,36291.977156,0.835722
3,Exp 2: meer features,"Overall Qual, Gr Liv Area, Neighborhood, Total...",5,2000,0.005,adaptive,0.0001,l2,23340.626581,1.397336e+09,37380.964085,0.825715
4,"Exp 1: zelfde features, andere hyperparameters","Overall Qual, Gr Liv Area, Neighborhood",3,2000,0.005,adaptive,0.0001,l2,24938.995954,1.534780e+09,39176.267698,0.808572
5,Initiële run,"Overall Qual, Gr Liv Area, Neighborhood",3,1000,0.010,invscaling,0.0001,l2,25603.894429,1.582183e+09,39776.660403,0.802660


In [14]:
# Dit blok laat alleen de belangrijkste scores netjes zien
samenvatting = resultaten_df[["Experiment", "Aantal features", "max_iter", "eta0", "learning_rate", "MAE", "RMSE", "R2"]].copy()
display(samenvatting.round(3))


,Experiment,Aantal features,max_iter,eta0,learning_rate,MAE,RMSE,R2
0,Exp 4: bijna alle features,10,2000,0.005,adaptive,21603.155,35857.674,0.840
1,Exp 5: bijna alle features + andere epochs/lr,10,3000,0.003,adaptive,21618.715,35875.812,0.839
2,Exp 3: nog meer features,7,2000,0.005,adaptive,22113.765,36291.977,0.836
3,Exp 2: meer features,5,2000,0.005,adaptive,23340.627,37380.964,0.826
4,"Exp 1: zelfde features, andere hyperparameters",3,2000,0.005,adaptive,24938.996,39176.268,0.809
5,Initiële run,3,1000,0.010,invscaling,25603.894,39776.660,0.803


In [15]:
# Dit blok pakt het beste experiment op basis van de laagste RMSE
beste_model = resultaten_df.iloc[0]
display(beste_model)


Experiment                                Exp 4: bijna alle features
Features           Garage, Overall Qual, Gr Liv Area, Total Bsmt ...
Aantal features                                                   10
max_iter                                                        2000
eta0                                                           0.005
learning_rate                                               adaptive
alpha                                                         0.0001
penalty                                                           l2
MAE                                                     21603.154972
MSE                                                 1285772801.26437
RMSE                                                    35857.674231
R2                                                           0.83963
Name: 0, dtype: object

## Korte conclusie

- De **initiële run** werkte al redelijk goed.
- Alleen de **hyperparameters aanpassen** gaf al een kleine verbetering.
- **Meer relevante features toevoegen** gaf de grootste winst.
- Het **beste model** in deze notebook is:
  - **Exp 4: bijna alle features**
  - met `max_iter = 2000`
  - `eta0 = 0.005`
  - `learning_rate = "adaptive"`

### Waarom inspireerde elk experiment het volgende?
- Omdat Exp 1 iets beter was dan de initiële run, wist ik dat **epochs/learning rate** invloed hadden.
- Daarna heb ik **meer features toegevoegd**, omdat er waarschijnlijk nog bruikbare informatie ontbrak.
- Toen dat weer beter werkte, heb ik bijna alle bruikbare features geprobeerd.
- Daarna testte ik nóg een andere combinatie van epochs/lr (Exp 5), maar die was net iets minder goed dan Exp 4.

### Eindconclusie
Het model werd duidelijk beter:
- **Lagere fout** (MAE en RMSE daalden)
- **Hogere verklaarde variantie** (`R2` steeg)

Dus: **meer goede features + nette hyperparameters = beter lineair regressiemodel**.
